# Incremental FHIR API ingestion

Fetch one resource incrementally, follow FHIR Bundle next links, and preserve each original API Bundle response as a raw JSON envelope in the Lakehouse Files area. Run this notebook sequentially for Patient, Encounter, Observation and Condition. The pipeline must pass resource_type, watermark_utc and batch_id.

In [ ]:
# Fabric pipeline parameters (mark this cell as the Parameters cell in Fabric if required)
resource_type = "Patient"
watermark_utc = ""  # empty = rolling 72-hour bootstrap window
batch_id = ""
page_size = 100
overlap_minutes = 5
max_pages = 10000

import json, time, uuid
from datetime import datetime, timezone, timedelta
from urllib.parse import urlparse
import requests

BASE_URL = "https://hapi.fhir.org/baseR4"
ALLOWED_RESOURCES = {"Patient", "Encounter", "Observation", "Condition"}
if resource_type not in ALLOWED_RESOURCES:
    raise ValueError(f"Unsupported resource_type: {resource_type}")
if not batch_id:
    batch_id = str(uuid.uuid4())

def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def parse_utc(value):
    dt = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)

if not watermark_utc:
    watermark_utc = (datetime.now(timezone.utc) - timedelta(hours=72)).strftime("%Y-%m-%dT%H:%M:%SZ")
search_from = parse_utc(watermark_utc) - timedelta(minutes=int(overlap_minutes))
params = {"_lastUpdated": "ge" + search_from.strftime("%Y-%m-%dT%H:%M:%SZ"), "_count": int(page_size)}

session = requests.Session()
session.headers.update({"Accept": "application/fhir+json"})
base_parts = urlparse(BASE_URL)
url = f"{BASE_URL}/{resource_type}"
visited = set()
page_number = 0
raw_root = f"Files/raw/fhir/{resource_type.lower()}/extraction_date={datetime.now(timezone.utc).strftime('%Y-%m-%d')}/batch_id={batch_id}"
page_audit = []

while url:
    parts = urlparse(url)
    if parts.scheme != base_parts.scheme or parts.netloc != base_parts.netloc or not parts.path.startswith(base_parts.path):
        raise ValueError(f"Rejected pagination URL outside configured FHIR base: {url}")
    if url in visited:
        raise RuntimeError(f"Pagination loop detected: {url}")
    if page_number >= int(max_pages):
        raise RuntimeError(f"Exceeded max_pages={max_pages}")
    visited.add(url)

    last_error = None
    response = None
    for attempt in range(3):
        try:
            response = session.get(url, params=params, timeout=60)
            response.raise_for_status()
            break
        except requests.RequestException as exc:
            last_error = exc
            if attempt < 2:
                time.sleep([2, 5, 10][attempt])
    if response is None or not response.ok:
        raise RuntimeError(f"FHIR request failed: {last_error}")

    bundle = response.json()
    if bundle.get("resourceType") != "Bundle":
        raise ValueError(f"Expected Bundle, received {bundle.get('resourceType')}")
    extraction_ts = utc_now()
    envelope = {
        "ingestion_metadata": {
            "resource_type": resource_type,
            "batch_id": batch_id,
            "page_number": page_number,
            "extraction_timestamp": extraction_ts,
            "api_url_or_params": response.url,
            "http_status": response.status_code,
            "source_total": bundle.get("total")
        },
        "fhir_bundle": bundle
    }
    raw_path = f"{raw_root}/page_{page_number:05d}.json"
    notebookutils.fs.put(raw_path, json.dumps(envelope, ensure_ascii=False), True)
    page_audit.append({
        "page_number": page_number, "raw_path": raw_path,
        "request_url": response.url, "http_status": response.status_code,
        "entry_count": len(bundle.get("entry", []) or []),
        "extraction_timestamp": extraction_ts
    })
    next_url = next((item.get("url") for item in bundle.get("link", []) or [] if item.get("relation") == "next"), None)
    url = next_url
    params = None
    page_number += 1

if not page_audit:
    print(f"No pages returned for {resource_type}; no watermark should be advanced.")
else:
    audit_schema = "resource_type string, batch_id string, page_number long, raw_path string, request_url string, http_status int, entry_count long, extraction_timestamp string"
    audit_rows = [(resource_type, batch_id, x["page_number"], x["raw_path"], x["request_url"], x["http_status"], x["entry_count"], x["extraction_timestamp"]) for x in page_audit]
    audit_df = spark.createDataFrame(audit_rows, audit_schema)
    audit_df.write.format("delta").mode("append").saveAsTable("metadata_ingestion_log")
    print(f"Persisted {len(page_audit)} raw pages for {resource_type}, batch {batch_id}")
    print("Watermark is intentionally not committed here; commit it after Bronze validation.")
